# 问题一：单点往返运输能力与货箱组批模型

## 1 问题

| 项目 | 内容 |
| --- | --- |
| **研究对象** | 临时调度中心 O01 与 15 个服务区 S001–S015 之间的**单点直接往返架次** O01 → Sᵢ → O01 |
| **任务 1** | 计算三种机型（A/B/C）在各服务区执行单点往返时的**最大安全载荷** |
| **任务 2** | 在货箱不可拆、每箱只装一次、满足**载质量 / 装载体积 / 返航安全能量余量**的条件下，给出各服务区的货箱组批方案 |
| **任务 3** | 完成全部 80 箱交付前提下，综合**往返架次数、总运输能耗、累计作业时间**三指标优化组批，并说明指标之间的**权衡 / 优先关系** |
| **任务 4** | 讨论**返航安全余量 $\rho$** 变化对各机型最大安全载荷与组批结果的影响（敏感性分析） |
| **输入数据** | 节点坐标与海拔、30 m DEM、机型参数（质量、航程、能量、速度、效率、作业时间）、逐箱货箱清单 |
| **硬约束** | 不跨服务区组批；货箱不可拆分；每箱只安排一次；$\sum质量 \leq 机型最大载货质量且 \leq 能量允许载荷$；$\sum体积 \leq 可用装载体积$；架次总能耗 $\leq (1-\rho)\cdot E^{use}$ |
| **本问不考虑** | 实体无人机数量、共享电池周转、配送时限（时限在问题二才成为约束，本问仅用于确定首批货箱的排序偏好） |
| **论文必须呈现** | 最大安全载荷表（15×3）；逐箱组批方案表；三指标结果与权衡分析；$\rho$ 敏感性曲线与结论 |

## 2 技术路线

```text
地理数据（节点 / DEM） ──▶ 航段几何模型 ──▶ 三时段时间模型 ─────────┐
                              │（距离、像元遍历、避障扫描）       │
                              ▼                              ▼
                   能耗模型（等效航程 + 爬升做功） ──▶ 最大安全载荷（单调性检查 + 二分搜索）
                                                            │
                                                            ▼
                   货箱组批：装载模式枚举 ──▶ 多目标动态规划（Pareto 剪枝） ──▶ 策略选择与权衡
                                                    │
                                                    ▼
             返航安全余量 ρ 的敏感性扫描（载荷 / 架次 / 能耗 / 时间 / 可行域）
```

> 航段几何模型：（距离、像元遍历、巡航海拔）

**三条路线上的每个环节都对应论文中的一小节**


## 3 题目统一规则的数学化（附录 2 的逐条解读）

**（1）架次结构。** 一个架次 $p=(g,i)$ 由机型 $g\in\{A,B,C\}$ 执行，路径为

$$
O_{01}\rightarrow S_i\rightarrow O_{01}.
$$

去程携带载荷 $q$（该批货箱总质量），回程空载 $q=0$。

**（2）巡航海拔与作业高度。** 航段取两节点间水平直线；**计划巡航海拔**

$$
H_i^c=\max[\mathrm{DEM}(\text{航段所经像元})]+50\,\mathrm{m}.
$$

$O_{01}$ 作业高度取其地面海拔 $z_O$；服务区作业高度取 $z_{S_i}+30\,\mathrm{m}$。于是：

$$
h_{\mathrm{out}}^{+}=H_i^c-z_O,\qquad
h_{\mathrm{out}}^{-}=H_i^c-(z_{S_i}+30),
$$

$$
h_{\mathrm{ret}}^{+}=H_i^c-(z_{S_i}+30),\qquad
h_{\mathrm{ret}}^{-}=H_i^c-z_O.
$$

**（3）三阶段飞行时间。** 每个航段按“爬升—巡航—下降”串行计算：

$$
t_{gij}
=
\frac{h_{ij}^{+}}{v_g^{\uparrow}}
+
\frac{d_{ij}}{v_g^{c}}
+
\frac{h_{ij}^{-}}{v_g^{\downarrow}}.
$$

**（4）等效航程与水平能耗。** 题目给出

$$
L_g(q)
=
L_g^0
-
\left(L_g^0-L_g^F\right)
\left(\frac{q}{Q_g}\right)^{3/2},
\qquad
0\leq q\leq Q_g.
$$

其物理含义是：机型 $g$ 携带载荷 $q$ 时，**一组可用能量 $E_g^{use}$ 可以支撑的水平巡航距离**。因此单位水平距离的能耗为 $E_g^{use}/L_g(q)$，航段水平巡航能耗

$$
E_{gij}^{hor}(q)
=
\frac{d_{ij}}{L_g(q)}\,E_g^{use}.
$$

指数 $3/2$ 源于旋翼飞行器诱导功率 $P\propto (mg)^{3/2}$ 的动理理论关系，**说明载荷对航程的侵蚀是加速的**：前一半载荷只损失约 $35\%$ 的可用航程差，后一半损失其余 $65\%$。

**（5）爬升附加能耗。** 爬升阶段需克服重力做功，附件给出爬升能耗效率 $\eta_g^{\uparrow}=0.72$、下降能耗效率 $0$（不单独计下降附加能耗）：

$$
E_{gij}^{up}(q)
=
\frac{(m_g^0+q)\,g_0\,h_{ij}^{+}}
{\eta_g^{\uparrow}\cdot 3.6\times 10^6}
\quad (\mathrm{kWh}),
\qquad
g_0=9.81\ \mathrm{m/s^2},
$$

其中 $m_g^0$ 为含电池空载总质量。中继无人机部分爬升附加能耗按上述相同关系计算，质量取计划起飞总质量，也印证了这一口径。

**（6）架次总能耗与返航安全余量。** 单点往返架次的能耗是去程（载荷 $q$）与回程（空载）之和：

$$
E_{gi}(q)
=
\underbrace{E_{gi}^{hor}(q)+E_{gi}^{up}(q;h_{out}^{+})}_{\text{去程}}
+
\underbrace{E_{gi}^{hor}(0)+E_{gi}^{up}(0;h_{ret}^{+})}_{\text{回程}}
\leq
(1-\rho_g)\,E_g^{use}.
$$

$E_{gi}(q)$ 关于 $q$ 严格单调递增且为凸函数（$1/L_g(q)$ 递增凸，爬升项线性），这两条性质分别是二分求最大安全载荷与“等架次下均衡分载能耗最小”的理论依据。

**（7）累计作业时间。** 一个装载 $n$ 箱的架次的作业时间为

$$
T_{gi}(n)
=
t_g^{prep}
+
n\,t_g^{load}
+
t_{gi}^{fly}
+
t_g^{hand}
+
n\,t_g^{box},
\qquad
t_{gi}^{fly}
=
t_{g,O\to S_i}
+
t_{g,S_i\to O}.
$$

问题一不考虑实体无人机与电池调度，故三指标均为逐架次求和：架次数

$$
N=\sum_p 1,
$$

总能耗

$$
E=\sum_p E_{gi}(q_p),
$$

累计作业时间

$$
T=\sum_p T_{gi}(n_p).
$$

## 4 运行环境、字体与图表主题

论文图表要求：中文宋体、英文与数字 Times New Roman、300 dpi 导出，低饱和统一配色。下面的 `setup_fonts()` 会依次尝试 Windows / macOS / Linux 的字体路径，找不到宋体时退化到系统可用的中文衬线字体并给出提示（不影响计算）。中英文混排的字体回退依赖 matplotlib ≥ 3.6；依赖库仅 numpy / pandas / scipy / matplotlib / openpyxl（读 GeoTIFF 备用路径需 tifffile）。

In [2]:
import math, json, time, itertools, bisect, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import font_manager
from matplotlib.colors import LightSource, LinearSegmentedColormap, ListedColormap, Normalize, BoundaryNorm
from matplotlib.patches import Rectangle, Polygon as MplPolygon, FancyBboxPatch
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection, PolyCollection
import matplotlib.patheffects as pe

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 120)
T_START = time.time()

# ---------- 路径；数据默认与 notebook 同目录 ----------
BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / '结果_问题一'
OUT_DIR.mkdir(exist_ok=True)

def find_file(name):
    """先在当前目录找，找不到再递归子目录找。"""
    p = BASE_DIR / name
    if p.exists():
        return p
    hits = list(BASE_DIR.rglob(name))
    if hits:
        return hits[0]
    raise FileNotFoundError(f'未找到文件 {name}：请将其放到与 notebook 相同目录（或其子目录）')


# ────────── 字体 ──────────
def setup_fonts():
    cands = ['C:/Windows/Fonts/simsun.ttc', 'C:/Windows/Fonts/times.ttf', 'C:/Windows/Fonts/timesbd.ttf',
             '/System/Library/Fonts/Supplemental/Songti.ttc', '/System/Library/Fonts/Supplemental/Times New Roman.ttf',
             '/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf']
    for c in cands:
        if Path(c).exists():
            try:
                font_manager.fontManager.addfont(c)
            except Exception:
                pass
    names = {f.name for f in font_manager.fontManager.ttflist}
    cn = next((n for n in ['SimSun', '宋体', 'Songti SC', 'Noto Serif CJK SC', 'Noto Serif CJK JP', 'Source Han Serif SC',
                           'Noto Sans CJK SC', 'Noto Sans CJK JP', 'Microsoft YaHei', 'SimHei', 'WenQuanYi Micro Hei'] if n in names), None)
    en = next((n for n in ['Times New Roman', 'Liberation Serif', 'DejaVu Serif', 'STIXGeneral'] if n in names), 'serif')
    fam = [en] + ([cn] if cn else [])
    plt.rcParams['font.family'] = fam
    plt.rcParams['font.serif'] = fam
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['mathtext.fontset'] = 'stix'
    return en, cn


EN_FONT, CN_FONT = setup_fonts()
print(f'英文/数字字体: {EN_FONT}    中文字体: {CN_FONT}' +
      ('' if CN_FONT == 'SimSun' else '（提示：未检测到宋体，已使用替代中文字体）'))


# ────────── 统一视觉主题 ──────────
PAL = dict(indigo='#1F3A5F', teal='#2A9D8F', coral='#E76F51', gold='#E9C46A', plum='#6D597A',
           sky='#8AB6D6', grey='#8D99AE', ink='#2B2D42', mist='#E4E6EB', sand='#B8AFA2')
TYPE_COLOR = {'A': '#2A9D8F', 'B': '#1F3A5F', 'C': '#E76F51'}  # 机型
MAT_COLOR = {'医疗物资': '#C8553D', '饮用水': '#3A7CA5', '应急食品': '#E9B44C', '生活卫生用品': '#7B6D8D'}  # 物资
MAT_ABBR = {'医疗物资': 'MED', '饮用水': 'WAT', '应急食品': 'FOD', '生活卫生用品': 'HYG'}
CMAP_SEQ = LinearSegmentedColormap.from_list('seq', ['#2A9D8F', '#1F3A5F', '#E76F51'])  # 青绿→靛蓝→珊瑚

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.edgecolor': '#A4A6E9', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#E4E6EB', 'grid.linewidth': 0.6,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8.5, 'legend.frameon': False,
    'tick.color': '#A4A6E9', 'ytick.color': '#A4A6E9', 'axes.labelcolor': '#2B2D42', 'text.color': '#2B2D42',
    'lines.linewidth': 1.6, 'patch.linewidth': 0.8,
})

HALO = [pe.withStroke(linewidth=2.5, foreground='white')]  # 文字白色描边，避免压在地形上看不清


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png')
    print(f'🖼️ 已保存 {OUT_DIR.name}/{name}.png')

英文/数字字体: Times New Roman    中文字体: Songti SC（提示：未检测到宋体，已使用替代中文字体）


KeyError: 'tick.color is not a valid rc parameter (see rcParams.keys() for a list of valid parameters)'

## 5 数据读取与数据台账

附件 Excel 采用“块状”排版（标题行 → 表头行 → 数据行 → 空行），普通 `read_excel` 无法直接使用，因此写一个通用的 `read_block()`：按块标题定位，以下一行为表头，读到空行为止。DEM 优先读取 `.mat`（`scipy.io` 即可，无需 GDAL/rasterio），失败时回退到 GeoTIFF。

读入后立刻做三件事：① 字段改成简短英文名便于建模；② 核对 DEM 在节点位置的高程与附件海拔是否一致（判断两套数据是否同源、有无粗差）；③ 汇总逐箱清单，确认 80 箱、4 类物资、质量/体积口径。

In [ ]:
def read_block(df, title):
    """从块状 Excel 中读取名为 title 的数据块（标题行 → 表头行 → 数据行 → 空行）。"""
    col0 = df[0].astype(str).str.strip()
    idx = int(col0.index[col0 == title][0])
    header = [str(h).strip() for h in df.iloc[idx + 1].tolist()]
    rows = []
    for k in range(idx + 2, len(df)):
        r = df.iloc[k]
        if r.isna().all():
            break
        rows.append(r.tolist())
    out = pd.DataFrame(rows, columns=header).dropna(axis=1, how='all')
    return out.loc[:, [c for c in out.columns if c != 'nan']].reset_index(drop=True)